# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook demonstrates how to explore and process the FAIR² Mass Spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.### Dataset SourceThe dataset is described by a [Croissant schema](https://mlcommons.org/croissant/), accessible via a URL.

In [ ]:
# Install the mlcroissant library in your environment.!pip install -U mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport json# Croissant schema URL for FAIR² datasetcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'print('Loading mlcroissant Dataset...')dataset = mlc.Dataset(croissant_url)metadata = dataset.metadata # Not a dict; keep as object.print(f"{metadata.name}: {metadata.description}")print(f"Published: {getattr(metadata, 'datePublished', 'n/a')}")

## 2. Data OverviewReview available record sets, fields, and their IDs.Each data entity in Croissant is uniquely identified by an `@id`. We'll print all available record sets, fields, and columns by their `@id` for reference.

In [ ]:
# Print dataset record sets and each field's @idprint('RecordSet entities available in the dataset:')record_sets = metadata.recordSetif isinstance(record_sets, dict):    record_sets = [record_sets]if not record_sets or len(record_sets)==0:    print('No record sets embedded directly in top-level metadata. Searching in distribution(s).')# If no recordSet at top, check via distributionsif not record_sets or len(record_sets)==0:    if hasattr(metadata, 'distribution') and metadata.distribution:        all_record_sets = []        for dist in metadata.distribution:            if hasattr(dist, 'recordSet') and dist.recordSet:                rs = dist.recordSet                if isinstance(rs, dict): rs = [rs]                all_record_sets.extend(rs)        record_sets = all_record_setsif not record_sets:    raise Exception('No record sets discovered in the Croissant metadata.')for idx, rec_set in enumerate(record_sets):    print(f"RecordSet {idx+1} @id: {rec_set['@id']}")    # Print details of fields/columns for this record set    if 'field' in rec_set:        fields = rec_set['field']        if isinstance(fields, dict):            fields = [fields]        print('  Fields:')        for field in fields:            f_name = field.get('name', '[no name]')            print(f"    @id: {field['@id']} -- {f_name}")            # Show columns for each field if any            if 'column' in field:                cols = field['column']                if isinstance(cols, dict): cols = [cols]                print('      Columns:')                for col in cols:                    print(f"        @id: {col['@id']} -- {col.get('name', '[no name]')}")    elif 'column' in rec_set:        cols = rec_set['column']        if isinstance(cols, dict): cols = [cols]        print('  Columns:')        for col in cols:            print(f"    @id: {col['@id']} -- {col.get('name', '[no name]')}")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis. Use the `@id` for the record set and fields obtained from the overview above.

In [ ]:
# Select all record set @id's for extractionrecord_set_ids = [rec_set['@id'] for rec_set in record_sets]dataframes = {}# Load records for each record set into DataFramefor rec_id in record_set_ids:    print(f'Loading records for RecordSet @id: {rec_id}')    try:        records = list(dataset.records(record_set=rec_id))        df = pd.DataFrame(records)        dataframes[rec_id] = df        print(f'  Loaded {len(df)} records')    except Exception as e:        print(f'  Failed to load records for {rec_id}: {e}')# Show columns of the first available dataframe (assuming first record set is main table)main_record_set = record_set_ids[0]if main_record_set in dataframes:    print('Available columns in main record set DataFrame:')    print(dataframes[main_record_set].columns.tolist())    display(dataframes[main_record_set].head())else:    print('Main record set was not loaded successfully.')

## 4. Exploratory Data Analysis (EDA)Let's perform some simple analyses such as filtering by a numeric field, normalizing, and optionally grouping by another column.Replace `<numeric_field_id>` and `<group_field_id>` below with the actual field or column `@id`/names as seen in section 2 or 3.

In [ ]:
# Choose numeric field. For this dataset, let's guess a field such as# 'coverage_pct' or 'MW' (Molecular Weight) as sample numeric columns.df = dataframes.get(main_record_set)display_cols = df.columns.tolist() if df is not None else []possible_numeric_fields = [col for col in display_cols if any(x in col.lower() for x in ['coverage', 'mw', 'weight', 'abundance', 'count', 'number', 'score', 'pI'])]if possible_numeric_fields:    numeric_field = possible_numeric_fields[0]else:    numeric_field = display_cols[0] if display_cols else Noneif numeric_field is None or numeric_field not in df.columns:    print('No suitable numeric field found for EDA.')else:    print(f'Using numeric field: {numeric_field}')        # Filter records by threshold    try:        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10    except:        threshold = 10        df_numeric = pd.to_numeric(df[numeric_field], errors='coerce')    filtered_df = df[df_numeric > threshold]    print(f"Filtered records where {numeric_field} > {threshold:.2f}:")    display(filtered_df.head())        # Normalize the selected numeric field    filtered_df = filtered_df.copy()    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()        print(f"Normalized '{numeric_field}' for filtered records:")    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())    # Try grouping by another field (e.g. description, accession, or experiment condition)    group_candidates = [col for col in df.columns if col != numeric_field and (        any(x in col.lower() for x in ['description', 'group', 'condition', 'sample', 'experiment', 'accession'])    )]    group_field = group_candidates[0] if group_candidates else df.columns[0] if len(df.columns) > 1 else None    if group_field:        print(f"Grouping by field: {group_field}")        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()        print('Grouped data (mean):')        display(grouped_df.head())

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.We'll plot a histogram of the numeric field and (if grouping possible) a bar plot of means by group.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as snsif numeric_field and numeric_field in df.columns:    plt.figure(figsize=(7,4))    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)    plt.title(f"Distribution of {numeric_field}")    plt.xlabel(numeric_field)    plt.show()    # Grouped bar plot    if group_field:        sorted_grp = grouped_df.sort_values(numeric_field, ascending=False).head(10)        plt.figure(figsize=(10,5))        sns.barplot(x=group_field, y=numeric_field, data=sorted_grp)        plt.title(f"Mean {numeric_field} by {group_field} (top 10)")        plt.xlabel(group_field)        plt.ylabel(f"Mean {numeric_field}")        plt.xticks(rotation=60, ha='right')        plt.show()

## 6. Conclusion- We've loaded and explored the FAIR² dataset using its Croissant schema.- The `mlcroissant` library enables programmatic access to metadata and record content.- From record set and field IDs, we extracted and visualized protein-related data.Further in-depth EDA or machine learning models can build on these extraction and transformation steps. Visit the [mlcroissant docs](https://mlcommons.org/croissant/) for more advanced operations.